# MechAudit: an end-to-end workflow

This notebook introduces MechAudit through a finite one-buyer mechanism-design problem. The small model keeps every step inspectable and lets the full optimal-face analysis finish quickly.

We will follow the library's three distinct workflows:

1. **Audit** a supplied mechanism without making any optimality claim.
2. **Design** an optimal randomized direct mechanism and independently certify it.
3. **Analyze and assess** the optimal face and a separate candidate mechanism.

Positive payments always mean money paid by the agent to the designer. Allocations are probability distributions over explicit social outcomes.

## Setup

From a released package, install with `python -m pip install mechaudit`. From a source checkout, use `python -m pip install -e .`. Then select that environment as the notebook kernel.

In [ ]:
import pandas as pd

from mechaudit import (
    DSIC,
    DirectMechanism,
    ExpectedRevenue,
    ExPostIR,
    FiniteEnvironment,
    NoSubsidies,
    PrivateValues,
)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 1. Declare the economic environment

The buyer's value is either 1 or 2 with equal probability. The social outcome is explicitly either `no_sale` or `sale`. `PrivateValues` is a semantic declaration: MechAudit does not infer private values by inspecting a callback.

In [ ]:
buyer_values = {"low": 1.0, "high": 2.0}

environment = FiniteEnvironment(
    agents={"buyer": ["low", "high"]},
    outcomes=["no_sale", "sale"],
    prior={"low": 0.5, "high": 0.5},
    values=PrivateValues(
        lambda _agent, own_type, outcome: buyer_values[own_type] if outcome == "sale" else 0.0
    ),
)

assert environment.n_agents == 1
assert environment.n_profiles == 2
assert environment.n_outcomes == 2
environment.profiles

## 2. Audit a supplied mechanism

First consider a menu that always sells, charging 1 after a low report and 2 after a high report. It is individually rational, but the high type can report `low` and save 1. A fixed audit should expose that witness without attaching an objective or an optimal face.

In [ ]:
discriminatory_menu = DirectMechanism(
    allocation={
        "low": {"no_sale": 0.0, "sale": 1.0},
        "high": {"no_sale": 0.0, "sale": 1.0},
    },
    payments={"low": 1.0, "high": 2.0},
)

audit = environment.audit(discriminatory_menu, (DSIC(), ExPostIR()))
assert not audit.passed
assert abs(audit.epsilon_ic - 1.0) < 1e-9
print(audit.summary())
audit.ic.worst_deviations(k=3)

In [ ]:
# Every check retains its economic family, labels, status, slack, and witness fields.
audit.to_dataframes()["ic"]

## 3. Design and certify an optimal mechanism

Now maximize expected revenue subject to dominant-strategy incentive compatibility, ex-post individual rationality, and no subsidies. MechAudit compiles a sparse maximization LP, solves it with HiGHS, reconstructs the mechanism, and runs the independent audit before returning a `DesignSolution`.

In [ ]:
problem = environment.design(
    objective=ExpectedRevenue(),
    requirements=(DSIC(), ExPostIR(), NoSubsidies()),
)
solution = problem.solve()

assert solution.audit.passed
assert abs(solution.objective_value - 1.0) < 1e-9
print(solution.summary())

In [ ]:
mechanism_tables = solution.mechanism.to_pandas()
print("Allocation probabilities")
display(mechanism_tables.allocation) if "display" in globals() else print(
    mechanism_tables.allocation
)
print("Payments to the designer")
mechanism_tables.payments

## 4. Separate selected activity from invariant activity

A solver returns one optimal point. A zero slack at that point is only local activity. Full optimal-face analysis maximizes each selected inequality slack and then minimizes every nonuniversal slack to distinguish constraints that are universally active, sometimes active, or never active across all optimal mechanisms.

In [ ]:
selected_activity = solution.to_dataframes()["selected_activity"]
selected_activity.head(10)

In [ ]:
face = solution.analyze_optima(
    level="full",
    objective_delta=0.0,
    n_jobs=1,
)

assert not face.failures
print(face.summary())
face_activity = face.to_dataframes()["optimal_face_activity"]
face_activity.head(12)

The reported classifications are relative to the stored tolerances and effective objective delta. Full analysis is row-wise and can be much more expensive than the baseline solve on larger models.

## 5. Assess a candidate against the solved problem

A candidate assessment aligns a separate mechanism with the same domain, audits the design requirements, evaluates the same maximize-convention objective, and reports its gap from the certified optimum. This pooling mechanism always sells at price 0.5: it is feasible, but suboptimal.

In [ ]:
pooling_candidate = DirectMechanism(
    allocation={
        "low": {"no_sale": 0.0, "sale": 1.0},
        "high": {"no_sale": 0.0, "sale": 1.0},
    },
    payments={"low": 0.5, "high": 0.5},
    environment=environment,
)

assessment = problem.assess(
    pooling_candidate,
    solution=solution,
    analyze_optimal_face=False,
)

assert assessment.is_feasible
assert not assessment.is_optimal
assert abs(assessment.absolute_optimality_gap - 0.5) < 1e-9
print(assessment.summary())
assessment.to_pandas()

## 6. Export reproducible numerical artifacts

Reports expose deterministic JSON and labeled, long-form pandas tables. JSON artifacts retain fingerprints, solver metadata, tolerances, and failed auxiliary tasks; they are numerical evidence rather than executable snapshots when the environment contains Python callbacks.

In [ ]:
solution_tables = solution.to_dataframes()
face_tables = face.to_dataframes()
solution_json = solution.to_json()

assert '"report_kind": "design_solution"' in solution_json
pd.DataFrame(
    {
        "artifact": ["audit", "solution", "optimal face", "assessment"],
        "available_tables": [
            ", ".join(sorted(audit.to_dataframes())),
            ", ".join(sorted(solution_tables)),
            ", ".join(sorted(face_tables)),
            "one-row assessment summary",
        ],
    }
)

## Where to go next

- Replace the one-buyer environment with multiple agents or a correlated `JointPrior`.
- Use `InterdependentValues` with `ExPostIC` when values depend on the full true profile.
- Start large post-optimal studies with `level="fast"` and selected constraint families.
- Record the MechAudit and HiGHS versions, model fingerprint, tolerances, and effective delta in research outputs.

The standalone scripts in `examples/` cover Myerson design, bilateral trade, correlated interdependent values, multidimensional types, Crémer–McLean diagnostics, and revenue-equivalent optima.